# UniverSeg Complexity Analysis

This notebook measures the computational complexity, inference speed, and GPU memory usage of the UniverSeg model.

Simulates a full volume of **128 × 128×128 2D images** for comparable benchmarking with PromptUNet.

In [1]:
# ! pip install torch thop

In [2]:
import torch
import time
import sys
import os
import warnings

In [3]:
# # Ensure we can import from the benchmark_models/UniverSeg if needed
# try:
#     from universeg import universeg
#     print("Successfully imported universeg from default path.")
# except ImportError:
#     current_dir = os.getcwd()
#     universeg_path = os.path.abspath(os.path.join(current_dir, '..', 'benchmark_models', 'UniverSeg'))

#     if not os.path.exists(universeg_path):
#         universeg_path = os.path.abspath(os.path.join(current_dir, 'evaluation', 'benchmark_models', 'UniverSeg'))

#     if os.path.exists(universeg_path):
#         if universeg_path not in sys.path:
#             sys.path.append(universeg_path)
#         try:
#             from universeg import universeg
#             print(f"Successfully imported universeg from {universeg_path}")
#         except ImportError:
#             print(f"Failed to import universeg even after adding {universeg_path} to path.")
#     else:
#         print(f"universeg not found. Please ensure it is installed or run in its environment.")

In [ ]:
# !git clone https://github.com/JJGO/UniverSeg.git
# %cd UniverSeg
# !pip install -e .

Cloning into 'UniverSeg'...
remote: Enumerating objects: 230, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 230 (delta 4), reused 8 (delta 3), pack-reused 216 (from 1)
Receiving objects: 100% (230/230), 35.26 MiB | 15.99 MiB/s, done.
Resolving deltas: 100% (100/100), done.
/content/UniverSeg
Obtaining file:///content/UniverSeg
  Preparing metadata (setup.py) ... done
  Running setup.py develop for universeg


In [5]:
from universeg import universeg
print("UniverSeg imported successfully!")

UniverSeg imported successfully!


In [6]:
# Try to import thop for MACs/FLOPs
try:
    from thop import profile
    HAS_THOP = True
    print("thop is installed. GFLOPs calculation enabled.")
except ImportError:
    HAS_THOP = False
    warnings.warn("thop is not installed. GFLOPs calculation will be skipped. Install it via 'pip install thop'")

thop is installed. GFLOPs calculation enabled.


In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Initialize model
print("Loading UniverSeg model...")
try:
    # Using pretrained=False to speed up initialization since we just want to measure complexity
    model = universeg(pretrained=False).to(device)
    model.eval()
    print("Model loaded successfully.")
except Exception as e:
    print(f"Error loading model: {e}")

Using device: cuda
Loading UniverSeg model...
Model loaded successfully.


In [8]:
# Define input dimensions according to UniverSeg requirements
B = 1   # Batch size
S = 1   # Support set size
H = 128 # Height
W = 128 # Width
NUM_SLICES = 128  # Slices per volume

print(f"Input shape (Target):  (Batch={B}, C=1, H={H}, W={W})")
print(f"Input shape (Support): (Batch={B}, Support_Set={S}, C=1, H={H}, W={W})")
print(f"Volume simulation:     {NUM_SLICES} slices of {H}x{W}")

# Dummy data for warm-up / single-pass tests
target_image   = torch.randn(B, 1, H, W).to(device)
support_images = torch.randn(B, S, 1, H, W).to(device)
support_labels = torch.randint(0, 2, (B, S, 1, H, W)).float().to(device)

Input shape (Target):  (Batch=1, C=1, H=128, W=128)
Input shape (Support): (Batch=1, Support_Set=1, C=1, H=128, W=128)
Volume simulation:     128 slices of 128x128


## Model Complexity (GFLOPs)

In [9]:
if HAS_THOP:
    # Calculate FLOPs using thop
    macs, params = profile(model, inputs=(target_image, support_images, support_labels), verbose=False)
    # 1 MAC is generally considered as 2 FLOPs (one multiply, one add)
    gflops = (macs * 2) / 1e9
    print(f"Parameters: {params / 1e6:.2f} M")
    print(f"GFLOPs (approx): {gflops:.2f}")
else:
    # Basic param count if thop is missing
    params = sum(p.numel() for p in model.parameters())
    print(f"Parameters: {params / 1e6:.2f} M")
    print("GFLOPs: N/A (Install 'thop' module to automatically calculate FLOPs)")

Parameters: 0.52 M
GFLOPs (approx): 6.44


## Inference Speed (Volume Simulation)

Simulates processing a full volume of `NUM_SLICES` 128×128 2D images, one slice at a time.

Each slice uses its own random input tensor to avoid caching effects.
GPU is synchronised after every slice for precise per-slice timing.

In [10]:
# ---- Warm-up phase ----
print('Warming up model...')
with torch.no_grad():
    for _ in range(10):
        _ = model(target_image, support_images, support_labels)

if device.type == 'cuda':
    torch.cuda.synchronize()
print('Warm-up complete.\n')


# ---- Generate random volume data (128 distinct slices) ----
print(f'Generating {NUM_SLICES} random slices...')
vol_targets  = [torch.randn(B, 1, H, W).to(device) for _ in range(NUM_SLICES)]
vol_supports = [torch.randn(B, S, 1, H, W).to(device) for _ in range(NUM_SLICES)]
vol_labels   = [torch.randint(0, 2, (B, S, 1, H, W)).float().to(device) for _ in range(NUM_SLICES)]


# ---- Timed volume inference ----
print(f'Running volume inference ({NUM_SLICES} slices)...\n')

if device.type == 'cuda':
    torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.no_grad():
    for i in range(NUM_SLICES):
        _ = model(vol_targets[i], vol_supports[i], vol_labels[i])
        if device.type == 'cuda':
            torch.cuda.synchronize()

end_time = time.perf_counter()

total_volume_time = end_time - start_time
avg_slice_time    = total_volume_time / NUM_SLICES
throughput         = NUM_SLICES / total_volume_time

print(f'--- Volume Inference Results ({NUM_SLICES} slices of {H}x{W}) ---')
print(f'Total volume inference time : {total_volume_time * 1000:.2f} ms')
print(f'Average per-slice time      : {avg_slice_time * 1000:.2f} ms')
print(f'Throughput                  : {throughput:.2f} slices/sec')

Warming up model...
Warm-up complete.

Generating 128 random slices...
Running volume inference (128 slices)...

--- Volume Inference Results (128 slices of 128x128) ---
Total volume inference time : 613.34 ms
Average per-slice time      : 4.79 ms
Throughput                  : 208.69 slices/sec


## GPU Memory Usage

In [11]:
if device.type == 'cuda':
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    # Run one full volume pass to capture peak memory
    with torch.no_grad():
        for i in range(NUM_SLICES):
            _ = model(vol_targets[i], vol_supports[i], vol_labels[i])

    max_mem    = torch.cuda.max_memory_allocated() / (1024 ** 3)  # GB
    max_mem_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)  # MB
    print(f"Peak GPU Memory Allocated: {max_mem_mb:.2f} MB ({max_mem:.3f} GB)")
else:
    print("GPU Memory Usage: N/A (running on CPU)")

Peak GPU Memory Allocated: 74.33 MB (0.073 GB)
